# Phase 2 — Gemma Pivot Alignment & Labeling

Clean replication notebook for the Gemma run.

**Input:** Phase 1 `pivot_candidates_gemma.jsonl` (110 candidates expected from the current run).  
**Output:** `phase2_labeled_gemma.jsonl`, ready for Phase 3 after manual review.

This version deliberately freezes the Qwen labeling protocol rather than redesigning it for Gemma:
1. same pivot lexicon,
2. same ground-truth keyword table (with the Qwen audit corrections already incorporated),
3. first qualifying pivot only,
4. automatic pre/post labeling,
5. fresh Gemma-specific manual review of uncertain cases,
6. provenance and QA checks before export.

**Important:** do not copy Qwen manual overrides into Gemma. Gemma's generations must be reviewed independently.

## 1. Configuration

In [19]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/notebooks/gunjalgupta/phase-1-gemma/__results__.html
/kaggle/input/notebooks/gunjalgupta/phase-1-gemma/__notebook__.ipynb
/kaggle/input/notebooks/gunjalgupta/phase-1-gemma/__output__.json
/kaggle/input/notebooks/gunjalgupta/phase-1-gemma/custom.css
/kaggle/input/notebooks/gunjalgupta/phase-1-gemma/pivot_research_gemma/pivot_candidates_gemma.jsonl
/kaggle/input/notebooks/gunjalgupta/phase-1-gemma/.virtual_documents/__notebook_source__.ipynb
/kaggle/input/datasets/gunjalgupta/pivot-candidates-gemma/pivot_candidates_gemma.jsonl
/kaggle/input/datasets/gunjalgupta/phase1-outputs-jsonl/raw_outputs_gemma.jsonl


In [20]:
!pip install -q transformers

import json
import os
import random
import re

from collections import Counter, defaultdict
from pathlib import Path
from transformers import AutoTokenizer
from kaggle_secrets import UserSecretsClient


# =========================================================
# CONFIG
# =========================================================

PHASE1_PATH = "/kaggle/input/datasets/gunjalgupta/pivot-candidates-gemma/pivot_candidates_gemma.jsonl"

OUTPUT_DIR = Path("/kaggle/working/pivot_research_gemma")

PHASE2_AUTO_PATH = OUTPUT_DIR / "phase2_auto_gemma.jsonl"
PHASE2_FINAL_PATH = OUTPUT_DIR / "phase2_labeled_gemma.jsonl"
REVIEW_PATH = OUTPUT_DIR / "gemma_uncertain_review.jsonl"

GEMMA_MODEL_NAME = "google/gemma-2-2b-it"

MIN_PRE_PIVOT_TOKENS = 15

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# HUGGING FACE TOKEN
# =========================================================

user_secrets = UserSecretsClient()

try:
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    print("HF token loaded successfully.")
except Exception as e:
    HF_TOKEN = None
    print("WARNING: Could not load HF_TOKEN.")
    print(e)


# =========================================================
# JSONL HELPERS
# =========================================================

def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )


# =========================================================
# LOAD PHASE 1 DATA
# =========================================================

if not os.path.exists(PHASE1_PATH):
    raise FileNotFoundError(
        f"Phase 1 file not found:\n{PHASE1_PATH}"
    )

records = load_jsonl(PHASE1_PATH)

print(f"\nLoaded {len(records)} Phase-1 records.")
print(f"Input file: {PHASE1_PATH}")


# =========================================================
# INSPECT MODEL METADATA
# =========================================================

for key in ("model_name", "model", "checkpoint"):

    vals = sorted({
        str(r.get(key))
        for r in records
        if r.get(key)
    })

    if vals:
        print(f"{key}: {vals}")


# =========================================================
# LOAD GEMMA TOKENIZER
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(
    GEMMA_MODEL_NAME,
    token=HF_TOKEN,
    use_fast=True
)

if not getattr(tokenizer, "is_fast", False):
    raise RuntimeError(
        "This notebook requires a fast tokenizer because "
        "Phase 2 uses offset mappings for character-to-token alignment."
    )


print("\nTokenizer loaded successfully.")
print("Checkpoint:", GEMMA_MODEL_NAME)
print("Fast tokenizer:", tokenizer.is_fast)


# =========================================================
# OUTPUT PATH CHECK
# =========================================================

print("\nPhase 2 outputs will be written to:")

print("Automatic labels:")
print(PHASE2_AUTO_PATH)

print("\nFinal reviewed labels:")
print(PHASE2_FINAL_PATH)

print("\nUncertain review file:")
print(REVIEW_PATH)

HF token loaded successfully.

Loaded 111 Phase-1 records.
Input file: /kaggle/input/datasets/gunjalgupta/pivot-candidates-gemma/pivot_candidates_gemma.jsonl
model_name: ['google/gemma-2-2b-it']

Tokenizer loaded successfully.
Checkpoint: google/gemma-2-2b-it
Fast tokenizer: True

Phase 2 outputs will be written to:
Automatic labels:
/kaggle/working/pivot_research_gemma/phase2_auto_gemma.jsonl

Final reviewed labels:
/kaggle/working/pivot_research_gemma/phase2_labeled_gemma.jsonl

Uncertain review file:
/kaggle/working/pivot_research_gemma/gemma_uncertain_review.jsonl


## 2. Frozen pivot lexicon

This is intentionally the same 18-pattern lexicon used for Qwen. We do **not** add Gemma-specific phrases after seeing its outputs, because that would weaken the cross-model replication.

In [21]:
PIVOT_PATTERNS = [
    (r"\bwait,\s", "explicit_correction"),
    (r"\bwait\u2014", "explicit_correction"),
    (r"\bactually,\s+i\b", "explicit_correction"),
    (r"\bactually,\s+no\b", "explicit_correction"),
    (r"\bactually,\s+that\b", "explicit_correction"),
    (r"\bactually,\s+(?:no|let|i|that|this|the|we|my|on)", "explicit_correction"),
    (r"\bi made an error", "explicit_correction"),
    (r"\bi was wrong", "explicit_correction"),
    (r"\bthat('s| is) (wrong|incorrect|not right)", "explicit_correction"),
    (r"\bi need to correct", "explicit_correction"),
    (r"\bno,\s+wait", "explicit_correction"),
    (r"\bi('ve| have) made a mistake", "explicit_correction"),
    (r"\blet me reconsider", "restart"),
    (r"\blet me recalculate", "restart"),
    (r"\blet me re-?read", "restart"),
    (r"\bon second thought", "hedging"),
    (r"\bhold on", "hedging"),
    (r"\bhmm,\s", "hedging"),
]

PIVOT_RE = re.compile("|".join(f"({p})" for p, _ in PIVOT_PATTERNS), re.IGNORECASE)
print(f"Frozen lexicon: {len(PIVOT_PATTERNS)} patterns")

Frozen lexicon: 18 patterns


## 3. Exact character → token alignment

Unlike the older notebook's decode-and-replay mapping, this version uses the tokenizer's native `offset_mapping`. This avoids silent alignment errors caused by whitespace or subword decoding.

In [22]:
def detect_pivots(text):
    enc = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )
    offsets = enc["offset_mapping"]
    input_ids = enc["input_ids"]

    detections = []

    for match in PIVOT_RE.finditer(text):
        pivot_type = "unknown"
        for group_idx, (_, ptype) in enumerate(PIVOT_PATTERNS, start=1):
            if match.group(group_idx) is not None:
                pivot_type = ptype
                break

        char_start, char_end_exclusive = match.span()

        overlapping = [
            i for i, (s, e) in enumerate(offsets)
            if e > char_start and s < char_end_exclusive
        ]
        if not overlapping:
            continue

        token_start = min(overlapping)
        token_end = max(overlapping)

        if token_start < MIN_PRE_PIVOT_TOKENS:
            continue

        detections.append({
            "char_start": char_start,
            "char_end": char_end_exclusive,
            "phrase": match.group(),
            "pivot_type": pivot_type,
            "token_start": token_start,
            "token_end": token_end,
            "pre_pivot_tokens": token_start,
            "total_tokens": len(input_ids),
        })

    return detections

## 4. Frozen ground-truth table

This table incorporates the corrections already discovered during the Qwen audit, including the corrected `mis_08`, `mis_09`, `math_02`–`math_09`, and `math_36` entries. It is frozen before Gemma labeling.

In [23]:
GROUND_TRUTH = {
    "mis_01": ["roosters don't lay", "rooster", "can't lay", "cannot lay", "doesn't lay"],
    "mis_02": ["noah", "not columbus", "columbus didn't"],
    "mis_03": ["don't bury", "survivors", "can't bury survivors", "you don't bury"],
    "mis_04": ["one hour", "60 minutes", "1 hour"],
    "mis_05": ["second", "2nd", "last place"],
    "mis_06": ["you have 4", "you took 4", "4 oranges"],
    "mis_07": ["7", "seven cows"],
    "mis_08": ["candle", "light the candle", "candle first"],
    "mis_09": ["50 cents", "$0.50", "0.50", "fifty cents"],
    "mis_10": ["all", "every", "12", "all of them"],

    "math_01": ["40 mph", "40mph", "harmonic"],
    "math_02": ["93.75", "not equal", "lower", "less"],
    "math_03": ["150", "$150", "not 200", "not back to"],
    "math_04": ["4 days", "four days"],
    "math_05": ["44%", "44 percent"],
    "math_06": ["5 minutes", "five minutes", "still 5"],
    "math_07": ["3 draws", "3", "correct", "true", "guaranteed"],
    "math_08": ["96%", "decrease", "4% less", "not the same", "not equal"],
    "math_09": ["2 hours", "correct", "true"],
    "math_10": ["50%", "50 percent", "still 50", "independent", "no effect",
                "gambler's fallacy", "does not change", "same probability"],
    "math_11": ["7.5", "7.5 degrees"],
    "math_12": ["21", "twenty-one"],
    "math_13": ["25 feet", "25-foot"],
    "math_14": ["7 minutes", "seven minutes"],
    "math_15": ["128", "128 layers"],
    "math_16": ["8 feet", "8-foot"],
    "math_17": ["1/6", "16.7%", "six ways"],
    "math_18": ["30", "thirty"],
    "math_19": ["204", "204 squares"],
    "math_20": ["90", "ninety"],
    "math_21": ["friday"],
    "math_22": ["3.5", "three point five"],
    "math_23": ["62.5%", "5/8"],
    "math_24": ["15", "fifteen handshakes"],
    "math_25": ["no", "not necessarily", "12 is"],
    "math_26": ["impossible", "can't", "cannot", "never"],
    "math_27": ["2/15", "13.3%"],
    "math_28": ["24", "twenty-four"],
    "math_29": ["no", "not a right", "not right triangle"],
    "math_30": ["1331", "$1331"],
    "math_31": ["golden ratio", "1.618", "phi"],
    "math_32": ["488", "not 600"],
    "math_33": ["about 16", "0.159"],
    "math_34": ["24", "twenty-four zeros"],
    "math_35": ["301", "three hundred"],
    "math_36": ["good bet", "70%", "70.6%", "probable"],
    "math_37": ["one piece", "1 piece", "still connected"],
    "math_38": ["equal", "equals 1", "same as 1"],
    "math_39": ["5 cents", "$0.05", "0.05"],
    "math_40": ["50%", "50.7%", "about 50"],

    "logic_01": ["switch", "yes", "should switch", "2/3"],
    "logic_02": ["probably not", "less than 1%", "unlikely", "false positive"],
    "logic_03": ["librarian", "representativeness", "base rate"],
    "logic_04": ["50 cents and 5 cents", "half dollar", "50-cent"],
    "logic_05": ["10", "son is 10"],
    "logic_06": ["yes", "can be", "is a"],
    "logic_07": ["mixed box", "both", "apple-orange"],
    "logic_08": ["1/3", "33%", "one third"],
    "logic_09": ["fish bowl", "aquarium", "fishbowl"],
    "logic_10": ["3 weighings", "possible", "can be done"],
    "logic_11": ["yes", "must be", "there is always"],
    "logic_12": ["turn on", "leave one on", "heat"],
    "logic_13": ["no", "flawed", "not valid"],
    "logic_14": ["yes", "can guarantee", "strategy"],
    "logic_15": ["what would the other guard say", "same door", "other guard"],
    "logic_16": ["2/3", "two thirds", "0.667"],
    "logic_17": ["german", "fifth house"],
    "logic_18": ["40", "forty"],
    "logic_19": ["possible", "can be done", "not impossible"],
    "logic_20": ["photographer", "photo", "picture"],

    "mis2_01": ["8 goats", "have 8", "is 8", "= 8", "eight goats"],
    "mis2_02": ["5 candies", "have 5", "is 5", "= 5", "five candies"],
    "mis2_03": ["12 loaves", "are 12", "is 12", "= 12", "twelve loaves"],
    "mis2_04": ["fish can't drown", "fish cannot drown", "don't drown", "can't drown", "8 fish", "still 8"],
    "mis2_05": ["20 books", "are 20", "is 20", "= 20", "twenty books"],
    "mis2_06": ["no", "lost it", "fell out", "hole in", "don't have it", "no longer have"],
    "mis2_07": ["white", "polar bear", "north pole"],
    "mis2_08": ["don't bury", "you don't bury survivors", "not buried", "survivors aren't buried"],
    "mis2_09": ["9 marbles", "have 9", "is 9", "= 9", "nine marbles"],
    "mis2_10": ["14 apples", "are 14", "is 14", "= 14", "fourteen apples"],
    "mis2_11": ["have 4", "4 eggs", "took 4", "= 4"],
    "mis2_12": ["monopoly", "board game", "playing monopoly"],
    "mis2_13": ["4 months", "four months", "april", "june", "september", "november"],
    "mis2_14": ["still everest", "was always everest", "everest was still", "mount everest"],
    "mis2_15": ["walking", "he's walking", "not driving", "on foot"],
    "mis2_16": ["5 cows", "have 5", "is 5", "= 5", "five cows"],
    "mis2_17": ["have 2", "2 apples", "took 2", "= 2"],
    "mis2_18": ["13 pencils", "are 13", "is 13", "= 13", "thirteen pencils"],
}

## 5. Automatic labeler

Protocol is unchanged:

- wrong → right = `correction`
- right → wrong = `hallucination_pivot`
- wrong → wrong = `hallucination_pivot`
- right → right = `uncertain`

The automatic label is retained separately even after human review so the provenance is auditable.

In [24]:
def contains_any_keyword(text, keywords):
    t = text.lower()
    hits = [kw for kw in keywords if kw.lower() in t]
    return bool(hits), hits

def auto_label(record, pivot):
    pid = record["prompt_id"]
    if pid not in GROUND_TRUTH:
        return {
            "label": "uncertain",
            "reason": "missing_ground_truth",
            "pre_correct": None,
            "post_correct": None,
            "pre_hits": [],
            "post_hits": [],
        }

    text = record["generated_text"]
    pre = text[:pivot["char_start"]]
    post = text[pivot["char_start"]:]
    keywords = GROUND_TRUTH[pid]

    pre_correct, pre_hits = contains_any_keyword(pre, keywords)
    post_correct, post_hits = contains_any_keyword(post, keywords)

    if not pre_correct and post_correct:
        label, reason = "correction", "wrong_then_right"
    elif pre_correct and not post_correct:
        label, reason = "hallucination_pivot", "right_then_wrong"
    elif not pre_correct and not post_correct:
        label, reason = "hallucination_pivot", "wrong_then_wrong"
    else:
        label, reason = "uncertain", "right_then_right_or_keyword_ambiguity"

    return {
        "label": label,
        "reason": reason,
        "pre_correct": pre_correct,
        "post_correct": post_correct,
        "pre_hits": pre_hits,
        "post_hits": post_hits,
    }

## 6. Run automatic Phase 2

For comparability with Qwen, only the **first qualifying detected pivot** is analyzed. The notebook also records how many qualifying pivots were found so multi-pivot generations remain auditable.

In [25]:
results = []
no_detection = []

for r in records:
    detections = detect_pivots(r["generated_text"])
    if not detections:
        no_detection.append(r)
        continue

    pivot = detections[0]
    lab = auto_label(r, pivot)

    out = dict(r)
    out.update({
        "pivot_phrase": pivot["phrase"],
        "pivot_type": pivot["pivot_type"],
        "pivot_char_start": pivot["char_start"],
        "pivot_char_end": pivot["char_end"],
        "pivot_token_start": pivot["token_start"],
        "pivot_token_end": pivot["token_end"],
        "pre_pivot_tokens": pivot["pre_pivot_tokens"],
        "total_tokens": pivot["total_tokens"],
        "n_qualifying_pivots": len(detections),
        "auto_label": lab["label"],
        "label": lab["label"],
        "label_reason": lab["reason"],
        "label_method": "automatic",
        "pre_correct": lab["pre_correct"],
        "post_correct": lab["post_correct"],
        "pre_keyword_hits": lab["pre_hits"],
        "post_keyword_hits": lab["post_hits"],
        "phase2_model": GEMMA_MODEL_NAME,
    })
    results.append(out)

save_jsonl(results, PHASE2_AUTO_PATH)

print(f"Input candidates:       {len(records)}")
print(f"Successfully aligned:   {len(results)}")
print(f"No qualifying detection:{len(no_detection)}")
print("\nAutomatic label distribution:")
for k, v in Counter(r["label"] for r in results).most_common():
    print(f"  {k:<22} {v}")
print(f"\nSaved automatic snapshot: {PHASE2_AUTO_PATH}")

Input candidates:       111
Successfully aligned:   108
No qualifying detection:3

Automatic label distribution:
  hallucination_pivot    54
  uncertain              44
  correction             10

Saved automatic snapshot: /kaggle/working/pivot_research_gemma/phase2_auto_gemma.jsonl


## 7. QA checks before manual review

In [26]:
# 1) Every prompt must have ground truth.
missing_gt = sorted({r["prompt_id"] for r in results if r["prompt_id"] not in GROUND_TRUTH})
print("Missing ground truth:", missing_gt or "none")

# 2) Verify stored token span decodes to text overlapping the regex phrase.
alignment_failures = []
for r in results:
    enc = tokenizer(r["generated_text"], add_special_tokens=False)
    ids = enc["input_ids"][r["pivot_token_start"]:r["pivot_token_end"] + 1]
    decoded = tokenizer.decode(ids, skip_special_tokens=False).lower()
    phrase = r["pivot_phrase"].strip().lower()
    # Loose containment because tokenizer decoding may normalize spaces.
    if phrase and phrase not in decoded and decoded.strip() not in phrase:
        alignment_failures.append((r["prompt_id"], r.get("sample_idx"), phrase, decoded))

print("Alignment warnings:", len(alignment_failures))
for x in alignment_failures[:10]:
    print(" ", x)

# 3) Multi-pivot audit.
multi = [r for r in results if r["n_qualifying_pivots"] > 1]
print("Records with >1 qualifying pivot:", len(multi))

# 4) Category distribution.
def category(pid):
    if pid.startswith("math_"): return "math"
    if pid.startswith("logic_"): return "logic"
    return "misdirection"

print("\nBy category:")
for cat in ("misdirection", "math", "logic"):
    sub = [r for r in results if category(r["prompt_id"]) == cat]
    print(cat, len(sub), dict(Counter(r["label"] for r in sub)))

Missing ground truth: none
Alignment warnings: 0
Records with >1 qualifying pivot: 14

By category:
misdirection 57 {'uncertain': 30, 'hallucination_pivot': 20, 'correction': 7}
math 42 {'uncertain': 14, 'correction': 2, 'hallucination_pivot': 26}
logic 9 {'hallucination_pivot': 8, 'correction': 1}


## 8. Export uncertain cases for fresh Gemma review

Do not reuse the Qwen override log. Read each Gemma generation itself.

The review file contains the full text plus the automatic evidence. For each uncertain record, assign one of:

- `correction`
- `hallucination_pivot`
- `uncertain`

Use `uncertain` when the substantive status truly cannot be determined confidently.

In [27]:
uncertain = [r for r in results if r["label"] == "uncertain"]
save_jsonl(uncertain, REVIEW_PATH)

print(f"{len(uncertain)} uncertain Gemma records require manual review.")
print(f"Review file: {REVIEW_PATH}\n")

for r in uncertain:
    key = f'{r["prompt_id"]}#{r.get("sample_idx", "?")}'
    print("=" * 100)
    print("KEY:", key)
    print("PIVOT:", repr(r["pivot_phrase"]))
    print("AUTO:", r["label_reason"])
    print("PRE HITS:", r["pre_keyword_hits"])
    print("POST HITS:", r["post_keyword_hits"])
    print("-" * 100)
    print(r["generated_text"])
    print()

44 uncertain Gemma records require manual review.
Review file: /kaggle/working/pivot_research_gemma/gemma_uncertain_review.jsonl

KEY: mis_01#10
PIVOT: 'Wait, '
AUTO: right_then_right_or_keyword_ambiguity
PRE HITS: ["roosters don't lay", 'rooster']
POST HITS: ['rooster']
----------------------------------------------------------------------------------------------------
A rooster's eggs aren't real. Roosters don't lay eggs, hens do.  Wait, that's not how it works. Roosters are male chickens, hens are female chickens. Hens lay eggs. So, if the rooster is on the roof, and it's a hen that lays eggs, then the egg would roll in the direction the hen's feathers are facing. 



 


KEY: mis_03#5
PIVOT: 'wait, '
AUTO: right_then_right_or_keyword_ambiguity
PRE HITS: ["don't bury", 'survivors', "you don't bury"]
POST HITS: ['survivors']
----------------------------------------------------------------------------------------------------
This is a bit of a trick question.  Survivors of a crash are

## 9. Enter Gemma manual overrides

Fill this dictionary **after** reviewing the uncertain records printed above.

Example:

```python
MANUAL_OVERRIDES = {
    "mis_07#3": ("correction", "initial answer is wrong; after pivot Gemma explicitly resolves to 7"),
}
```

Keep a short reason for every override. This becomes the Gemma audit log.

In [29]:
# ============================================================
# 9. GEMMA MANUAL OVERRIDES
# ============================================================

MANUAL_OVERRIDES = {

    # --------------------------------------------------------
    # MISDIRECTION
    # --------------------------------------------------------

    "mis_01#10": (
        "hallucination_pivot",
        "The model correctly recognizes before the pivot that roosters do not lay eggs, "
        "but after the pivot introduces an incorrect hypothetical egg-and-hen explanation. "
        "Right-to-wrong pivot."
    ),

    "mis_03#5": (
        "uncertain",
        "The model correctly recognizes before the pivot that survivors are not buried. "
        "After the pivot it questions where survivors might be transported but does not "
        "substantively reverse the correct answer."
    ),

    # mis_05 is excluded conservatively because the underlying prompt is ambiguous.
    "mis_05#4": (
        "uncertain",
        "The underlying last-place-runner prompt is logically ambiguous, and the model "
        "does not reach a defensible unambiguous resolution."
    ),

    "mis_05#5": (
        "uncertain",
        "The underlying last-place-runner prompt is logically ambiguous, so this pivot "
        "cannot be reliably classified against a defensible ground truth."
    ),

    "mis_05#6": (
        "uncertain",
        "The underlying last-place-runner prompt is ambiguous; the generation considers "
        "multiple interpretations without establishing a defensible correction."
    ),

    "mis_05#7": (
        "uncertain",
        "The underlying last-place-runner prompt is ambiguous and the generation remains "
        "internally conflicted rather than reaching a clear correction."
    ),

    "mis_05#8": (
        "uncertain",
        "The underlying last-place-runner prompt is ambiguous. The model moves between "
        "second-place and last-place interpretations without a defensible ground truth."
    ),

    "mis_05#10": (
        "uncertain",
        "The underlying last-place-runner prompt is ambiguous and the model explicitly "
        "questions the wording rather than reaching a reliable substantive answer."
    ),

    "mis_05#12": (
        "uncertain",
        "The underlying last-place-runner prompt is ambiguous; the model repeats the "
        "last-place answer while expressing unresolved doubt."
    ),

    "mis_05#13": (
        "uncertain",
        "The underlying last-place-runner prompt is ambiguous. The model settles on "
        "second place, but the prompt itself does not support a clean ground-truth label."
    ),

    "mis_05#14": (
        "uncertain",
        "The underlying last-place-runner prompt is ambiguous and the generation moves "
        "between last and second-last interpretations without a defensible resolution."
    ),

    "mis_07#11": (
        "uncertain",
        "The model correctly states before the pivot that seven cows remain and explicitly "
        "returns to seven afterward. Right-to-right verification."
    ),

    "mis_07#13": (
        "uncertain",
        "The model correctly states that seven cows remain before the pivot and does not "
        "establish a different substantive answer afterward."
    ),

    "mis_07#14": (
        "correction",
        "The model's substantive pre-pivot answer is 13 cows remaining. After the pivot "
        "it recognizes that 'all but 7 are sold' means seven cows remain. Wrong-to-right correction."
    ),

    "mis_08#2": (
        "hallucination_pivot",
        "The model begins with the incorrect oil-lamp answer and, after reconsideration, "
        "moves to the stove rather than the intended candle-first answer. Wrong-to-wrong pivot."
    ),

    "mis_09#0": (
        "hallucination_pivot",
        "The model correctly derives a $0.50 ball price before the pivot but then incorrectly "
        "concludes that the valid $50.50 racket price does not add up. Right-to-wrong pivot."
    ),

    "mis_09#6": (
        "uncertain",
        "The model correctly derives $0.50 before the pivot, questions itself, and then "
        "repeats the same valid algebra and $0.50 result. Right-to-right verification."
    ),

    "mis_09#7": (
        "uncertain",
        "The model correctly derives $0.50 before the pivot but then repeatedly questions "
        "the result. The displayed continuation is truncated before a definitive alternative "
        "answer is established."
    ),

    "mis_09#12": (
        "hallucination_pivot",
        "The model correctly derives $0.50 before the pivot, then abandons the valid solution "
        "and concludes that no solution exists. Right-to-wrong pivot."
    ),

    "mis_10#1": (
        "uncertain",
        "The model ultimately recognizes that every month has at least 28 days and retains "
        "the answer of 12. The pivot is verification rather than a substantive correction."
    ),


    # --------------------------------------------------------
    # MATH
    # --------------------------------------------------------

    "math_01#0": (
        "uncertain",
        "The model correctly calculates 40 mph before the pivot and continues to identify "
        "40 mph as the correct answer afterward. Right-to-right verification."
    ),

    "math_06#1": (
        "uncertain",
        "The model correctly concludes that one machine takes five minutes per part before "
        "the pivot and reaffirms the same answer afterward."
    ),

    "math_09#6": (
        "hallucination_pivot",
        "The model's reasoning about distance and time is incorrect before the pivot and "
        "remains unresolved and incorrect afterward rather than reaching the correct "
        "two-hour meeting time."
    ),

    "math_09#10": (
        "hallucination_pivot",
        "The model correctly computes that 150 mph for two hours covers the 300-mile gap, "
        "but after the pivot incorrectly concludes that the trains should meet in less "
        "than two hours. Right-to-wrong pivot."
    ),

    "math_09#14": (
        "hallucination_pivot",
        "The model correctly computes 300/150 = 2 hours but nevertheless rejects the "
        "two-hour answer after the pivot. Right-to-wrong pivot."
    ),

    "math_14#9": (
        "hallucination_pivot",
        "The initial hourglass procedure is incorrect, and after recognizing a problem "
        "the model remains confused and never reaches a valid seven-minute strategy."
    ),

    "math_18#3": (
        "uncertain",
        "The model correctly obtains 30 items before the pivot. It questions the calculation "
        "but continues to use 10 divided by one-third, which yields the same correct quantity."
    ),

    "math_23#1": (
        "uncertain",
        "The model correctly computes 62.5% before the pivot, claims it was wrong, but then "
        "explicitly verifies that 5/8 equals 62.5%. Right-to-right verification."
    ),

    "math_23#12": (
        "uncertain",
        "The model correctly computes 62.5% before the pivot and, after reconsideration, "
        "returns to exactly the same 5/8 = 62.5% result."
    ),

    "math_23#13": (
        "hallucination_pivot",
        "The model correctly computes 62.5% before the pivot but then rejects the correct "
        "reasoning and does not recover a valid final percentage in the observed continuation."
    ),

    "math_25#5": (
        "correction",
        "The model initially makes the incorrect inference that divisibility by both 4 and 6 "
        "implies divisibility by 24. After the pivot it identifies that the LCM is 12 and "
        "correctly concludes that divisibility by 24 is not guaranteed."
    ),

    "math_28#6": (
        "uncertain",
        "The model correctly solves x = 24 before the pivot, checks the algebra afterward, "
        "and explicitly confirms that 24 is correct. Right-to-right verification."
    ),

    "math_28#11": (
        "uncertain",
        "The model correctly derives x = 24 before the pivot and verifies the solution "
        "afterward. No substantive correction occurs."
    ),

    "math_30#6": (
        "hallucination_pivot",
        "Although the model correctly computes $1331 before the pivot, it then rejects "
        "that correct compound-interest result in favor of the intuition that the answer "
        "should be closer to $1300. Right-to-wrong pivot."
    ),


    # --------------------------------------------------------
    # SECOND MISDIRECTION SET
    # --------------------------------------------------------

    "mis2_01#3": (
        "correction",
        "The model's substantive pre-pivot answer is 17 goats remaining. After the pivot "
        "it recognizes the 'all but 8' wording and correctly concludes that eight goats remain."
    ),

    "mis2_02#3": (
        "uncertain",
        "The model correctly answers five candies before the pivot and explicitly retains "
        "five afterward. Right-to-right verification."
    ),

    "mis2_02#4": (
        "uncertain",
        "The model correctly states five candies before the pivot, briefly questions the "
        "interpretation, and returns to five. Right-to-right."
    ),

    "mis2_02#5": (
        "uncertain",
        "The model initially gives the correct five-candy answer, becomes confused after "
        "the pivot, but does not establish a sufficiently clear alternative final answer."
    ),

    "mis2_02#14": (
        "uncertain",
        "The model already correctly interprets 'all but 5' as giving away ten and leaving "
        "five before the pivot. After the pivot it simply makes the remaining-five answer explicit."
    ),

    "mis2_09#0": (
        "uncertain",
        "The model correctly states nine marbles remain before the pivot. It entertains an "
        "alternative interpretation afterward but does not clearly replace the original "
        "correct answer with a stable incorrect one."
    ),

    "mis2_09#3": (
        "hallucination_pivot",
        "The model correctly states before the pivot that nine marbles remain, then "
        "misinterprets the wording as losing nine and moves to 21. Right-to-wrong pivot."
    ),

    "mis2_09#9": (
        "uncertain",
        "The model gives nine marbles before the pivot, briefly considers 21, but ultimately "
        "returns to and explicitly concludes nine. Right-to-right overall."
    ),

    "mis2_11#7": (
        "hallucination_pivot",
        "The model correctly states before the pivot that taking four eggs means having "
        "four eggs, but after the pivot incorrectly concludes that it has two. Right-to-wrong."
    ),

    "mis2_16#0": (
        "hallucination_pivot",
        "The model's substantive reasoning before the pivot incorrectly settles on seven cows. "
        "After recognizing that 'all but 5' means five survived, it nevertheless again concludes "
        "seven. Wrong-to-wrong pivot."
    ),
}


# ============================================================
# VALIDATE + APPLY
# ============================================================

VALID_LABELS = {
    "correction",
    "hallucination_pivot",
    "uncertain"
}

by_key = {
    f'{r["prompt_id"]}#{r.get("sample_idx", "?")}': r
    for r in results
}

print(f"Manual overrides supplied: {len(MANUAL_OVERRIDES)}")

missing = [
    key for key in MANUAL_OVERRIDES
    if key not in by_key
]

if missing:
    raise KeyError(
        "These override keys were not found:\n" +
        "\n".join(missing)
    )


for key, value in MANUAL_OVERRIDES.items():

    if not isinstance(value, (tuple, list)) or len(value) != 2:
        raise ValueError(
            f"{key}: override must be (label, reason)"
        )

    new_label, reason = value

    if new_label not in VALID_LABELS:
        raise ValueError(
            f"{key}: invalid label {new_label}"
        )

    r = by_key[key]

    # Preserve original automatic label for auditability
    if "auto_label" not in r or r["auto_label"] is None:
        r["auto_label"] = r.get("label")

    if "auto_label_reason" not in r:
        r["auto_label_reason"] = r.get("label_reason")

    # Apply human-reviewed label
    r["label"] = new_label
    r["label_reason"] = reason
    r["label_method"] = "human"
    r["human_verified"] = True


# ============================================================
# VERIFY
# ============================================================

print("\nApplied manual overrides successfully.\n")

override_counts = Counter(
    by_key[key]["label"]
    for key in MANUAL_OVERRIDES
)

print("Manual-review labels:")

for label in (
    "correction",
    "hallucination_pivot",
    "uncertain"
):
    print(
        f"  {label:<22}: "
        f"{override_counts[label]}"
    )


print("\nFinal labels for reviewed records:")

for key in MANUAL_OVERRIDES:

    r = by_key[key]

    print(
        f"{key:<15} "
        f"{str(r.get('auto_label')):<35} -> "
        f"{r['label']}"
    )


# ============================================================
# SAVE IMMEDIATELY
# ============================================================

save_jsonl(
    results,
    PHASE2_FINAL_PATH
)

print("\nSaved updated Phase-2 dataset:")
print(PHASE2_FINAL_PATH)

Manual overrides supplied: 44

Applied manual overrides successfully.

Manual-review labels:
  correction            : 3
  hallucination_pivot   : 13
  uncertain             : 28

Final labels for reviewed records:
mis_01#10       uncertain                           -> hallucination_pivot
mis_03#5        uncertain                           -> uncertain
mis_05#4        uncertain                           -> uncertain
mis_05#5        uncertain                           -> uncertain
mis_05#6        uncertain                           -> uncertain
mis_05#7        uncertain                           -> uncertain
mis_05#8        uncertain                           -> uncertain
mis_05#10       uncertain                           -> uncertain
mis_05#12       uncertain                           -> uncertain
mis_05#13       uncertain                           -> uncertain
mis_05#14       uncertain                           -> uncertain
mis_07#11       uncertain                           -> uncer

## 10. Final export + replication decision summary

This cell writes the final Gemma Phase-2 dataset and reports the number that matters for deciding whether to proceed to Phase 3: **usable correction examples with at least 30 pre-pivot tokens**.

The 30-token requirement matches the attention query window used in the paper.

In [30]:
save_jsonl(results, PHASE2_FINAL_PATH)

labels = Counter(r["label"] for r in results)
usable = [
    r for r in results
    if r["label"] in {"correction", "hallucination_pivot"}
    and r["pre_pivot_tokens"] >= 30
]
usable_labels = Counter(r["label"] for r in usable)

print("=" * 72)
print("GEMMA PHASE 2 — FINAL SUMMARY")
print("=" * 72)
print(f"Phase-1 candidates:          {len(records)}")
print(f"Aligned/labeled records:     {len(results)}")
print(f"Corrections:                 {labels['correction']}")
print(f"Hallucination pivots:        {labels['hallucination_pivot']}")
print(f"Uncertain:                   {labels['uncertain']}")
print(f"Usable for Phase 3 (>=30):   {len(usable)}")
print(f"  usable corrections:        {usable_labels['correction']}")
print(f"  usable hallucinations:     {usable_labels['hallucination_pivot']}")

print("\nUsable corrections by category:")
for cat in ("misdirection", "math", "logic"):
    n = sum(
        1 for r in usable
        if r["label"] == "correction" and category(r["prompt_id"]) == cat
    )
    print(f"  {cat:<14} {n}")

n_corr = usable_labels["correction"]
if n_corr >= 15:
    decision = "PROCEED to Phase 3 — correction count is strong enough for the planned replication."
elif n_corr >= 10:
    decision = "BORDERLINE but likely worth proceeding; interpret all Gemma estimates cautiously."
else:
    decision = "STOP before Phase 3 and reassess — too few usable corrections for a useful replication."

print("\nDecision:", decision)
print(f"\nFinal file: {PHASE2_FINAL_PATH}")

GEMMA PHASE 2 — FINAL SUMMARY
Phase-1 candidates:          111
Aligned/labeled records:     108
Corrections:                 13
Hallucination pivots:        67
Uncertain:                   28
Usable for Phase 3 (>=30):   76
  usable corrections:        13
  usable hallucinations:     63

Usable corrections by category:
  misdirection   9
  math           3
  logic          1

Decision: BORDERLINE but likely worth proceeding; interpret all Gemma estimates cautiously.

Final file: /kaggle/working/pivot_research_gemma/phase2_labeled_gemma.jsonl


## 11. Optional audit sample

Before Phase 3, inspect a fixed random sample from **every** final class—not only uncertain records. This is a safeguard against confident keyword-labeling errors like the stale-ground-truth issue found during the Qwen audit.

In [31]:
random.seed(42)

for label in ("correction", "hallucination_pivot", "uncertain"):
    subset = [r for r in results if r["label"] == label]
    print("\n" + "#" * 100)
    print(label.upper(), f"({len(subset)} records)")
    print("#" * 100)
    for r in random.sample(subset, min(5, len(subset))):
        print(f'\n{r["prompt_id"]}#{r.get("sample_idx", "?")} | '
              f'{r["pivot_type"]} | method={r["label_method"]}')
        print(r["generated_text"])
        print("-" * 100)


####################################################################################################
CORRECTION (13 records)
####################################################################################################

mis2_07#9 | explicit_correction | method=automatic
The problem states the house has four walls, each facing south.  This would mean the house is in the southern hemisphere, and the only land masses in the southern hemisphere are in the Southern Ocean and South America. We don't know anything about the house's location but we can assume that because the house is on land, it's on a geographical area where a bear is located. A bear could be any color, as bears exist in many parts of the world, including northern hemisphere, but they would not be able to survive in the Southern Ocean. However, I may be missing something. It doesn't say the bear was walking past the house. 

**Wait, the statement says a bear walks past the house, which could mean the bear was walking